# 🧪 Lab 3: The First VARIANT Boarding Pass (Progressive Structure & Native Exploration Foundation)

In this laboratory session, we open the doors to the Spark 4 terminal and migrate our semi-structured workflows onto the native `VARIANT` data type. Rather than forcing raw incoming payloads into rigid explicit schemas or leaving them as completely opaque text strings, we will use `parse_json` and `try_parse_json` to establish a flexible middle ground. We will evaluate progressive path extraction, safe casting boundaries, and schema aggregation across a series of drifting commercial travel offers.

### Step 1: Initialize Local Spark Workspace
We instantiate our localized standalone Spark session and verify that the core engine supports native variant capabilities.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from decimal import Decimal
import json

# Spin up local Spark context
spark = SparkSession.builder.getOrCreate()

print(f"Active Spark Engine Version: {spark.version}")

Active Spark Engine Version: 4.1.2


### Step 2: Ingest Raw JSON Strings into Native VARIANT
We ingest our multi-supplier travel payloads and parse them directly into a flexible `VARIANT` column using `parse_json`.

In [2]:
# Simulated landing stream containing deep nesting and extra vendor metrics
raw_travel_payloads = [
    ("BKG-001", '{"origin":"MAD","destination":"BCN","carrier":"IB","fare_family":"EconomyLight","totalAmount":87.50,"currency":"EUR","baggage":{"included":true,"allowance":["1PC"]},"legs":[{"seq":1,"from":"MAD","to":"BCN"}]}'),
    ("BKG-002", '{"origin":"CDG","destination":"JFK","carrier":"AF","fare_family":"EconomyClassic","totalAmount":520.00,"currency":"USD","baggage":{"included":true,"allowance":["1PC","23KG"]},"legs":[{"seq":1,"from":"CDG","to":"AMS"},{"seq":2,"from":"AMS","to":"JFK"}],"supplier_metadata":{"campaign":"SUMMER_SALE"}}')
]

bronze_df = spark.createDataFrame(raw_travel_payloads, ["booking_id", "raw_payload"])

# Parse raw text strings into native semi-structured variant types
variant_df = bronze_df.withColumn("payload_variant", F.parse_json(F.col("raw_payload")))

print("=== STEP 2: NAVIGABLE INGESTION LAYER CONFIGURATION ===")
variant_df.select("booking_id", "payload_variant").show(truncate=False)
variant_df.printSchema()

=== STEP 2: NAVIGABLE INGESTION LAYER CONFIGURATION ===
+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|booking_id|payload_variant                                                                                                                                                                                                                                                                                         |
+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|BKG-001   |{"

### Step 3: Progressive Extraction and Path Evaluation
We query our un-shredded variant column using path expressions via `variant_get` to selectively promote our stable business variables.

In [3]:
# Extract core parameters dynamically using json path selectors
extracted_variant_df = variant_df.select(
    "booking_id",
    F.variant_get(F.col("payload_variant"), "$.origin", "string").alias("origin"),
    F.variant_get(F.col("payload_variant"), "$.destination", "string").alias("destination"),
    F.variant_get(F.col("payload_variant"), "$.carrier", "string").alias("carrier"),
    F.variant_get(F.col("payload_variant"), "$.fare_family", "string").alias("fare_family"),
    F.variant_get(F.col("payload_variant"), "$.totalAmount", "decimal(10,2)").alias("total_amount"),
    F.variant_get(F.col("payload_variant"), "$.currency", "string").alias("currency"),
    F.variant_get(F.col("payload_variant"), "$.baggage.included", "boolean").alias("baggage_included")
)

print("=== STEP 3: SELECTIVE SILVER EXTRACTION VIEW ===")
extracted_variant_df.show(truncate=False)
extracted_variant_df.printSchema()

=== STEP 3: SELECTIVE SILVER EXTRACTION VIEW ===
+----------+------+-----------+-------+--------------+------------+--------+----------------+
|booking_id|origin|destination|carrier|fare_family   |total_amount|currency|baggage_included|
+----------+------+-----------+-------+--------------+------------+--------+----------------+
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50       |EUR     |true            |
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00      |USD     |true            |
+----------+------+-----------+-------+--------------+------------+--------+----------------+

root
 |-- booking_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- fare_family: string (nullable = true)
 |-- total_amount: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- baggage_included: boolean (nullable = true)



### Step 4: Injecting Controlled Ingestion Damage & Casting Anomalies
We test our error gates by exposing `try_parse_json` and `try_variant_get` to bad text payloads, text-wrapped numbers, and explicit null definitions.

In [4]:
damaged_stream = [
    ("BKG-MALFORMED", '{"origin":"MAD", "destination": "BCN",'), # Broken closing brace
    ("BKG-TYPE-DRIFT", '{"origin":"MAD","destination":"BCN","totalAmount":"cheap-ish","baggage":null}'), # Alphanumeric value under price
    ("BKG-ABSENT-BAG", '{"origin":"MAD","destination":"BCN","totalAmount":87.50}') # Absent baggage field
]

damaged_bronze_df = spark.createDataFrame(damaged_stream, ["booking_id", "raw_payload"])

# safe parse scanner
damaged_variant_df = damaged_bronze_df.withColumn("payload_variant", F.try_parse_json(F.col("raw_payload")))

radar_report_df = damaged_variant_df.select(
    "booking_id",
    F.try_variant_get(F.col("payload_variant"), "$.origin", "string").alias("origin"),
    F.try_variant_get(F.col("payload_variant"), "$.totalAmount", "decimal(10,2)").alias("total_amount"),
    F.is_variant_null(F.variant_get(F.col("payload_variant"), "$.baggage", "variant")).alias("is_explicit_json_null"),
    F.variant_get(F.col("payload_variant"), "$.baggage", "string").alias("raw_baggage_field")
)

print("=== STEP 4: ANOMALOUS RADAR ANATOMY ===")
radar_report_df.show(truncate=False)

=== STEP 4: ANOMALOUS RADAR ANATOMY ===
+--------------+------+------------+---------------------+-----------------+
|booking_id    |origin|total_amount|is_explicit_json_null|raw_baggage_field|
+--------------+------+------------+---------------------+-----------------+
|BKG-MALFORMED |NULL  |NULL        |false                |NULL             |
|BKG-TYPE-DRIFT|MAD   |NULL        |true                 |NULL             |
|BKG-ABSENT-BAG|MAD   |87.50       |false                |NULL             |
+--------------+------+------------+---------------------+-----------------+



### Step 5: Unpacking Flight Legs via Variant Explosion
We deploy the `variant_explode` table-valued function using a SQL lateral view correlation to cleanly un-nest itinerary leg arrays into distinct relational rows.

In [5]:
# Register DataFrame as temporary view to execute table-valued functions natively
variant_df.createOrReplaceTempView("variant_staging_table")

exploded_legs_df = spark.sql("""
    SELECT booking_id, pos, key, leg_variant
    FROM variant_staging_table,
    LATERAL variant_explode(variant_get(payload_variant, '$.legs', 'variant')) AS t(pos, key, leg_variant)
""")

itinerary_report_df = exploded_legs_df.select(
    "booking_id",
    "pos",
    F.variant_get(F.col("leg_variant"), "$.seq", "int").alias("segment_seq"),
    F.variant_get(F.col("leg_variant"), "$.from", "string").alias("leg_origin"),
    F.variant_get(F.col("leg_variant"), "$.to", "string").alias("leg_destination")
)

print("=== STEP 5: TRAVERSED FLIGHT SEGMENT LEDGER ===")
itinerary_report_df.show(truncate=False)

=== STEP 5: TRAVERSED FLIGHT SEGMENT LEDGER ===
+----------+---+-----------+----------+---------------+
|booking_id|pos|segment_seq|leg_origin|leg_destination|
+----------+---+-----------+----------+---------------+
|BKG-001   |0  |1          |MAD       |BCN            |
|BKG-002   |0  |1          |CDG       |AMS            |
|BKG-002   |1  |2          |AMS       |JFK            |
+----------+---+-----------+----------+---------------+



### Step 6: Discovered-Structure Schema Aggregation
We launch `schema_of_variant_agg` across our active dataset to build a structural report of our supplier channels.

In [6]:
discovered_schema_report = variant_df.select(F.schema_of_variant_agg(F.col("payload_variant"))).collect()[0][0]

print("=== STEP 6: AGGREGATED DISCOVERED SCHEMA PATTERN ===")
print(discovered_schema_report)

=== STEP 6: AGGREGATED DISCOVERED SCHEMA PATTERN ===
OBJECT<baggage: OBJECT<allowance: ARRAY<STRING>, included: BOOLEAN>, carrier: STRING, currency: STRING, destination: STRING, fare_family: STRING, legs: ARRAY<OBJECT<from: STRING, seq: BIGINT, to: STRING>>, origin: STRING, supplier_metadata: OBJECT<campaign: STRING>, totalAmount: DECIMAL(4,1)>


## 📊 Post-Lab Analysis: Evidence from the Engine Room

### 1. Binary Storage Optimization & Trailing Zero Erasure
* **The Precision Scale Shift:** In Step 6, `schema_of_variant_agg` outputs the finalized discovered structural footprint of `totalAmount` across our rows as a minimized `DECIMAL(4,1)`. Even though the original string literal for `BKG-001` explicitly defined a precision of `87.50` with trailing zeros, the underlying binary engine room systematically compressed the value down to `87.5` during the serialization phase.
* **Decoupled Extraction Realities:** This demonstrates that native binary `VARIANT` layout structures prioritize spatial minimization over original text formatting preservation. While your Step 3 extraction successfully casts the metric back up to a uniform `decimal(10,2)` layer for consumption (recovering `87.50`), the inner map only retains the active fractional precision required to mathematically represent the number.

### 2. Null Footprint Isolation and the Casting Blindspot
* **Explicit vs. Absent Verifications:** The telemetry from Step 4 perfectly substantiates the null discrimination thesis. For `BKG-TYPE-DRIFT` (where the provider transmitted an explicit literal `baggage: null`), `is_explicit_json_null` flags a positive `true`, whereas for `BKG-ABSENT-BAG` (where the key was missing entirely from the payload text), the engine flags it as `false`.
* **The Direct Casting Blindspot:** In that same radar step, the `raw_baggage_field` column attempts to parse `$.baggage` directly as a scalar `"string"`, which evaluates to a standard SQL `NULL` for all rows. This underscores a severe production verification risk: relying on plain path casting operators completely masks structural intent, blinding engineers by blending intentional literal nulls and un-transmitted paths into an identical state of nothingness unless isolated by `is_variant_null`.

### 3. Uniform Local Latency and Task Coordination Tax
* **The Execution Clock Constraint:** Reviewing the notebook's raw execution timers reveals a highly uniform latency profile across your silver operations, with Step 3 taking 11.1 seconds, Step 4 taking 11.7 seconds, Step 5 taking 11.7 seconds, and Step 6 taking 12.2 seconds to clear worker execution threads.
* **Fixed Coordination Costs:** Because your current sample space consists of only two or three row primitives, this clock speed is entirely independent of parsing computational complexity. The logs confirm that your runtime is paying a fixed local infrastructure tax for Catalyst compilation, JVM task serialization, and single-node coordination loops rather than actual text parsing overhead.